In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Union, List, Iterable
import re
import shutil
import numpy as np
import cv2
import pywt
from matplotlib.widgets import Slider
from typing import Union, List, Iterable, Dict
from ipywidgets import interact, IntSlider
from typing import Union, Optional, Tuple


def load_gray(path: Union[str, Path]) -> np.ndarray:
    """
    Load an image as float32 grayscale array (0-255).
    """
    p = Path(path)
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {p}")
    return img.astype(np.float32)

def show_intensity_histogram(
    img_path: Union[str, Path],
    bins: int = 256,
    title: str | None = None,
) -> Dict[str, float]:
    """
    Load a TIFF (or other) image and display its intensity histogram
    along with basic statistics.

    Returns a dict with min, max, mean, std.
    """
    img_path = Path(img_path)
    img = load_gray(img_path) 

    # Flatten to 1D for histogram and stats
    vals = img.ravel()

    stats = {
        "min": float(vals.min()),
        "max": float(vals.max()),
        "mean": float(vals.mean()),
        "std": float(vals.std()),
    }

    # Print stats
    print(f"Image: {img_path.name}")
    print(f"  min  = {stats['min']:.3f}")
    print(f"  max  = {stats['max']:.3f}")
    print(f"  mean = {stats['mean']:.3f}")
    print(f"  std  = {stats['std']:.3f}")

    # Plot histogram
    plt.figure(figsize=(6, 4))
    plt.hist(vals, bins=bins, range=(0, 255), edgecolor="black")
    plt.xlabel("Intensity value")
    plt.ylabel("Pixel count")
    if title is None:
        title = f"Intensity histogram: {img_path.name}"
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return stats


In [ ]:
stats = show_intensity_histogram(
    r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Plasma Data\251030\251030005_PROCESSED\251030005 001_proc.tif",
    bins=256,
)


In [ ]:
stats = show_intensity_histogram(
    r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Plasma Data\251030\251030005_PROCESSED\251030005 048_proc.tif",
    bins=256,
)


In [ ]:
stats = show_intensity_histogram(
    r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Plasma Data\251030\251030005_PROCESSED\251030005 055_proc.tif",
    bins=256,
)


In [ ]:
def horizontal_intensity_profile(img: np.ndarray) -> np.ndarray:
    """
    Compute mean grayscale intensity for each horizontal slice (row).
    
    Returns a 1D array of length = image height.
    """
    # img shape: (H, W)
    # mean over axis=1 --> one mean per row
    return img.mean(axis=1)

def plot_horizontal_profile(profile: np.ndarray, title: str = "") -> None:
    """
    Plot mean grayscale intensity vs pixel height index.
    Pixel height index is 1..H.
    """
    H = profile.shape[0]
    y_idx = np.arange(1, H + 1)  # 1..H

    plt.figure()
    plt.plot(y_idx, profile)
    plt.xlabel("Pixel height index")
    plt.ylabel("Mean grayscale intensity")
    plt.title(title if title else "Horizontal intensity profile")
    plt.xlim(1, H)
    plt.ylim(0, 255)
    plt.grid(True, alpha=0.3)
    plt.show()

def gather_frames(folder: Union[str, Path],
                  exts=(".tif", ".tiff", ".png", ".jpg", ".jpeg")) -> List[Path]:
    """
    Return a sorted list of image paths in 'folder' with allowed extensions.
    """
    folder = Path(folder)
    if not folder.is_dir():
        raise NotADirectoryError(folder)
    
    files = []
    for ext in exts:
        files.extend(folder.glob(f"*{ext}"))
    
    files = sorted(files)
    
    if not files:
        raise ValueError(f"No image files found in {folder}")
    
    return files

def compute_profiles_for_folder(folder: Union[str, Path]) -> tuple[list[np.ndarray], list[Path]]:
    """
    Load all frames in folder and compute horizontal intensity profiles.
    
    Returns:
        profiles: list of 1D np.ndarrays
        paths:    list of Path objects (same order)
    """
    paths = gather_frames(folder)
    profiles = []

    # Load first to get reference height
    first_img = load_gray(paths[0])
    ref_height = first_img.shape[0]

    profiles.append(horizontal_intensity_profile(first_img))

    for p in paths[1:]:
        img = load_gray(p)
        if img.shape[0] != ref_height:
            raise ValueError(
                f"Image {p} has different height {img.shape[0]} (expected {ref_height})."
            )
        profiles.append(horizontal_intensity_profile(img))

    return profiles, paths



In [ ]:
path = r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Tests\250930 Canned Air\jet_10 027 processed.tif"
img = load_gray(path)
profile = horizontal_intensity_profile(img)
plot_horizontal_profile(profile)

In [ ]:
def save_horizontal_profiles_for_folder(
    folder: Union[str, Path],
    output_subdir: str = "horizontal_profiles",
    dpi: int = 150,
    show_plots: bool = False,
) -> Path:
    """
    For each frame in `folder`, compute the horizontal intensity profile,
    generate a plot, and save it as an image.

    Args:
        folder: Path to folder containing image frames.
        output_subdir: Name of the subfolder (inside `folder`) to store plots.
        dpi: Resolution for saved plot images.
        show_plots: If True, also display each plot in the notebook.

    Returns:
        Path to the output directory containing the saved plot images.
    """
    folder = Path(folder)
    if not folder.is_dir():
        raise NotADirectoryError(folder)

    # Where to save the plots
    out_dir = folder / output_subdir
    out_dir.mkdir(exist_ok=True)

    # Get all frame paths
    frame_paths = gather_frames(folder)

    # Use first image to set reference height for consistent x limits
    first_img = load_gray(frame_paths[0])
    ref_height = first_img.shape[0]
    x_idx = np.arange(1, ref_height + 1)

    for i, p in enumerate(frame_paths):
        img = load_gray(p)

        if img.shape[0] != ref_height:
            raise ValueError(
                f"Image {p} has different height {img.shape[0]} (expected {ref_height})."
            )

        profile = horizontal_intensity_profile(img)

        # Plot
        plt.figure(figsize=(6, 4))
        plt.plot(x_idx, profile)
        plt.xlabel("Pixel height index")
        plt.ylabel("Mean grayscale intensity")
        plt.title(f"Horizontal intensity profile\nFrame {i+1}/{len(frame_paths)} - {p.name}")
        plt.xlim(1, ref_height)
        plt.ylim(0, 255)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        # Save figure
        out_path = out_dir / f"{p.stem}_profile.png"
        plt.savefig(out_path, dpi=dpi)

        if show_plots:
            plt.show()
        else:
            plt.close()

    print(f"Saved {len(frame_paths)} profile plots to: {out_dir}")
    return out_dir

In [ ]:
folder = r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Plasma Data\251030\251030005_PROCESSED"
out_dir = save_horizontal_profiles_for_folder(folder)

In [ ]:
def show_image_pixels(
    img: np.ndarray,
    *,
    title: str = "Image (pixel coordinates)",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
) -> None:
    """
    Plot an image with axes labeled in pixel coordinates.
    x-axis = column index (0..W-1), y-axis = row index (0..H-1)
    """
    if img.ndim != 2:
        raise ValueError("show_image_pixels expects a 2D grayscale image (H, W).")

    H, W = img.shape

    plt.figure()
    # extent maps array indices to axis coordinates in pixels
    plt.imshow(
        img,
        cmap="gray",
        origin="upper",
        extent=[0, W - 1, H - 1, 0],  # x: 0..W-1, y: 0..H-1 (top at y=0)
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="auto",
    )
    plt.xlabel("x pixel index (column)")
    plt.ylabel("y pixel index (row)")
    plt.title(title)
    plt.colorbar(label="intensity (0-255)")
    plt.tight_layout()
    plt.show()


def vertical_slice(
    img: np.ndarray,
    x: int,
    *,
    clip: bool = False,
) -> np.ndarray:
    """
    Return the vertical slice at column x as a 1D array of length H:
    slice[y] = img[y, x]

    If clip=True, x is clipped into [0, W-1]. Otherwise raises if out of range.
    """
    if img.ndim != 2:
        raise ValueError("vertical_slice expects a 2D grayscale image (H, W).")

    H, W = img.shape
    if clip:
        x = int(np.clip(x, 0, W - 1))
    else:
        if not (0 <= x < W):
            raise IndexError(f"x={x} out of bounds for image width W={W}")

    return img[:, x].copy()


def plot_vertical_slice(
    img: np.ndarray,
    x: int,
    *,
    title: Optional[str] = None,
    show_on_image: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract and plot intensity vs y for the vertical slice at x.

    Returns:
        y_idx: (H,) array of y pixel indices
        intens: (H,) array of intensities at that column
    """
    if img.ndim != 2:
        raise ValueError("plot_vertical_slice expects a 2D grayscale image (H, W).")

    H, W = img.shape
    intens = vertical_slice(img, x)
    y_idx = np.arange(H)

    # Optional: show where the slice is on the image
    if show_on_image:
        plt.figure()
        plt.imshow(
            img,
            cmap="gray",
            origin="upper",
            extent=[0, W - 1, H - 1, 0],
            interpolation="nearest",
            aspect="auto",
        )
        plt.axvline(x=x, linewidth=1.0)
        plt.xlabel("x pixel index (column)")
        plt.ylabel("y pixel index (row)")
        plt.title(f"Slice location at x = {x}")
        plt.tight_layout()
        plt.show()

    # Plot intensity vs y
    plt.figure()
    plt.plot(H-1 - y_idx, intens)
    plt.xlabel("y pixel index (row)")
    plt.ylabel("intensity (0-255)")
    plt.title(title if title is not None else f"Vertical slice intensity vs y (x={x})")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return y_idx, intens


# -------- Example usage --------
img = load_gray(r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260219\260219009 PROCESSED\260219009 054_proc.tif")
show_image_pixels(img, title="Frame 124 (tif) - pixel axes")
y, I = plot_vertical_slice(img, x=25, show_on_image=True)